Q Make one program which has a toy dataset and perform all the three types of regularization techniques by both ways i.e by scratch and also by sklearn in built functions at last compare them.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
np.random.seed(42)

In [2]:
# toy dataset
X = 2 * np.random.rand(100, 3)       # generates a 100x3 matrix value bw 0 and 2 ,100 samples 3 features
true_coef = np.array([3, 1.5, 0])    # y=3(x1) ​+ 1.5(x2) ​+ 0(x3)​ , means 3rd feature is intentionally useless to test regularizations ridge(shrink),lasso(remove it)
y = X @ true_coef + np.random.randn(100) * 0.5      # @ -> matrix multiplication

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [4]:
# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(X_test)

[[-0.76963497 -0.00361107 -0.68986506]
 [-1.16077562  1.04706726 -1.5076237 ]
 [ 1.82403548  0.94386904 -1.05899524]
 [ 0.61836511 -1.11607836  0.71948306]
 [ 0.30656376  1.50860934  0.73788149]
 [ 1.22820363  1.24874064 -1.75186812]
 [-0.86670907  0.73691746 -0.91976778]
 [ 0.45424864 -1.17929437 -1.21335374]
 [ 0.48166557 -1.68515773 -1.41035412]
 [-0.80467102 -1.39606654  1.46479907]
 [ 1.48941007  0.45769784  1.09479513]
 [ 1.47190923  0.97016034  0.5427774 ]
 [ 0.59149597  1.33998304 -0.07079802]
 [-0.37500401  1.6313607   1.70049122]
 [-0.6529957  -1.58975416  0.42546772]
 [-0.14723614  1.61369508  1.70472831]
 [-0.29347999  1.5105912  -1.28010177]
 [ 1.68995511  1.61017114  1.14388372]
 [-1.27732527 -1.60855849  0.52246696]
 [-1.36112758 -1.15996526  1.46963504]]


In [5]:
def add_bias(X):
    return np.c_[np.ones((X.shape[0], 1)), X]  # np.c_[A, B] -> concatenate column wise
X_train_b = add_bias(X_train)
X_test_b = add_bias(X_test)

In [6]:
# Regularization penalizes large weights!!
def gradient_descent(X, y, alpha=0.1, l1=0, l2=0, epochs=1000): # alpha -> learning rate , l1 l2 ->lasso and ridge coeff
    m, n = X.shape
    theta = np.zeros(n)             # initialize weights to zero
    for _ in range(epochs):         # repeats the updates many times
        predictions = X @ theta     # ycap(mx1)=X(mxn)*theta(nx1)
        errors = predictions - y
        gradient = (1/m) * (X.T @ errors) # gradient j =1/mX^T(X*theta-y)
        gradient[1:] += l2 * theta[1:] + l1 * np.sign(theta[1:]) # skip 0th term -> donot regularize bias term
        theta -= alpha * gradient
    return theta

In [7]:
# training model
lambda_val = .1
theta_ridge = gradient_descent(X_train_b, y_train, l2=lambda_val)
theta_lasso = gradient_descent(X_train_b, y_train, l1=lambda_val)
theta_elastic = gradient_descent(X_train_b, y_train, l1=0.05, l2=0.05)

In [8]:
ridge = Ridge(alpha=lambda_val)
lasso = Lasso(alpha=lambda_val)
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)

ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)
elastic.fit(X_train, y_train)

ElasticNet(alpha=0.1)

In [9]:
# Scratch predictions
y_pred_ridge_s = X_test_b @ theta_ridge
y_pred_lasso_s = X_test_b @ theta_lasso
y_pred_elastic_s = X_test_b @ theta_elastic

# Sklearn predictions
y_pred_ridge_sk = ridge.predict(X_test)
y_pred_lasso_sk = lasso.predict(X_test)
y_pred_elastic_sk = elastic.predict(X_test)

### Results

In [10]:
print("SCRATCH COEFFICIENTS")
print("Ridge:", theta_ridge)
print("Lasso:", theta_lasso)
print("Elastic:", theta_elastic)

SCRATCH COEFFICIENTS
Ridge: [4.37619296 1.55976777 0.73045254 0.02171036]
Lasso: [4.37619296 1.6127213  0.70592605 0.00773498]
Elastic: [4.37619296e+00 1.58413190e+00 7.18511322e-01 2.66146494e-03]


In [11]:
print("SKLEARN COEFFICIENTS")
print("Ridge:", np.concatenate(([ridge.intercept_], ridge.coef_)))
print("Lasso:", np.concatenate(([lasso.intercept_], lasso.coef_)))
print("Elastic:", np.concatenate(([elastic.intercept_], elastic.coef_)))

SKLEARN COEFFICIENTS
Ridge: [4.37619296 1.71944062 0.81105251 0.03987501]
Lasso: [4.37619296 1.61247225 0.70586529 0.        ]
Elastic: [4.37619296 1.58392815 0.71846432 0.        ]


In [12]:
print("Ridge Scratch:", mean_squared_error(y_test, y_pred_ridge_s))
print("Ridge Sklearn:", mean_squared_error(y_test, y_pred_ridge_sk))

Ridge Scratch: 0.26938917496551007
Ridge Sklearn: 0.15313412008267316


In [13]:
print("Lasso Scratch:", mean_squared_error(y_test, y_pred_lasso_s))
print("Lasso Sklearn:", mean_squared_error(y_test, y_pred_lasso_sk))

Lasso Scratch: 0.267846032543121
Lasso Sklearn: 0.2716197981930869


In [14]:
print("Elasticnet Scratch:", mean_squared_error(y_test, y_pred_elastic_s))
print("Elasticnet Sklearn:", mean_squared_error(y_test, y_pred_elastic_sk))

Elasticnet Scratch: 0.2741128011753616
Elasticnet Sklearn: 0.27550503176481567
